# ノートブック 04B — 日本市場センチメント分析

本ノートブックでは、YouTubeおよび@cosmeなどの公開されているソーシャルメディア上のコメントやレビューを対象に、
ユーザーの実際の感情（センチメント）を分析・定量化します。

## 分析手法

**アプローチ:**
本ノートブックでは、日本市場における美容消費者のセンチメントを2つのデータソースから
分析する。①YouTube日本語コメント（YouTube Data API v3経由）、②@cosmeレビューデータ
（公開ページのスクレイピング、アクセス可能な場合）。
日本語テキストの形態素解析にはMeCabおよびGiNZAを使用し、英語分析（NB04A）で
使用したVADERとは異なるNLPパイプラインを採用する。

**YouTubeデータ収集:**
- NB03で使用した日本語キーワードリスト（TIER_KEYWORDS_JP）を使用
- 各キーワードにつき上位10動画を取得
- 1動画あたり最大100コメントを抽出
- 収集項目：コメントテキスト、高評価数、投稿日、動画ID、キーワード
- 各コメントはキーワードを通じてティアに紐付け

**@cosmeデータ収集:**
- BeautifulSoupによる公開レビューページのスクレイピング
- 収集項目：星評価（1〜7）、レビューテキスト、投稿日、ブランド名、商品名
- robots.txtを遵守し、適切なリクエスト間隔を設定
- 個人情報（ユーザー名等）は収集・保持しない

**センチメントスコアリング:**
- YouTube：MeCab形態素解析 → 日本語感情極性辞書によるスコアリング
- @cosme：星評価をセンチメントの代理指標として使用
（5点満点を-1〜+1にスケール変換：(rating - 3) / 2）
- 最終スコアは高評価数による加重平均

**ティア別集計:**
- NB01〜03と同一のブランド→ティアマッピングを使用。
- ティア別加重平均センチメントスコアおよびコメント数（議論密度の代理指標）を算出。

**制限事項:**
- YouTubeの無料枠は1日10,000ユニット — 大規模取得には複数セッションが必要な場合あり
- @cosmeのスクレイピングはサイト構造の変更により予告なく機能しなくなる可能性あり
- 日本語感情極性辞書はドメイン（美容）特化型ではないため、
  業界固有の表現（「しっとり」「毛穴」等）のスコアリング精度に限界あり
- YouTubeは日本国内でも広く使用されているが、
  美容特化コミュニティとしての深度は@cosmeに劣る
- @cosmeのレビュアーは購入者に限定されないため、
  未使用者によるレビューが含まれる可能性あり
- 分析対象のブランドキーワードは網羅的なものではなく、
各ティアを代表する例として選定されています。

---

## ブランド→ティア対応表

| ティア | ブランド |
|--------|---------|
| **ラグジュアリー** | エスティローダー、クリニーク、ラ・メール、トム フォード ビューティ、ジョー マローン、ボビイ ブラウン、ディオール ビューティ、ゲラン、ジバンシィ ビューティ、ベネフィット、フレッシュ |
| **プレステージ** | ランコム、イヴ・サンローラン ビューティ、ジョルジオ アルマーニ ビューティ、資生堂、NARS、ドランク エレファント、クレ・ド・ポー ボーテ、コスメデコルテ、雪肌精、ダーマロジカ |
| **マスティージ** | CeraVe、ラ ロッシュ ポゼ、スキンシューティカルズ、ヴィシー、キュレル |
| **マス** | ロレアル パリ、メイベリン、NYX、ガルニエ、ダヴ、シンプル、ビオレ、カネボウ |

*注：ティア分類はNB01の収益セグメンテーションに基づく。
一部ブランド（ダーマロジカ、キュレル等）はティア境界に位置しており、
主要な消費者価格認知に基づいて分類している。*

---

## NB04Aとの対比

| 項目 | NB04A（英語圏） | NB04B（日本市場） |
|------|----------------|-----------------|
| データソース | YouTube（英語） | YouTube（日本語）+ @cosme |
| センチメント手法 | VADER | 感情極性辞書 / 星評価 |
| 形態素解析 | 不要 | MeCab / GiNZA |
| 対象市場 | 米国・英国・欧州 | 日本 |
| 言語 | 英語 | 日本語 |

In [ ]:
# ── 日本語フォント設定 ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Windows上の日本語フォント
plt.rcParams["font.family"] = "MS Gothic"

# 確認
print(fm.findSystemFonts(fontpaths=None, fontext='ttf'))

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────
!pip install ginza ja-ginza
!pip install google-api-python-client

In [ ]:
!pip install python-dotenv

In [ ]:
# ── Cell 2: インポートとセットアップ ───────────────────────
import spacy
import ginza
import pandas as pd
import matplotlib.pyplot as plt
from googleapiclient.discovery import build
import time
import json
import pickle
from datetime import datetime, timedelta
import sys
sys.path.append("../src")
from helpers import TIER_COLOURS, set_style, save_chart

set_style()

# ── YouTube接続 ────────────────────────────────────────────
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

# ── GiNZA/Sudachiモデル読み込み ────────────────────────────
nlp = spacy.load("ja_ginza")
print("GiNZA読み込み完了")

In [ ]:
# ── Cell 3: キーワードとジオ設定 ───────────────────────────

# ティア別日本語キーワード（NB03と同一）
TIER_KEYWORDS_JP = {
    "Luxury": [
        "エスティローダー", "クリニーク", "ラ・メール", "トム フォード ビューティ",
        "ジョー マローン", "ボビイ ブラウン", "ディオール ビューティ", "ゲラン",
        "ジバンシィ ビューティ", "ベネフィット", "フレッシュ"
    ],
    "Prestige": [
        "ランコム", "イヴ・サンローラン ビューティ", "ジョルジオ アルマーニ ビューティ",
        "資生堂", "NARS", "ドランク エレファント", "クレ・ド・ポー ボーテ",
        "コスメデコルテ", "雪肌精", "ダーマロジカ"
    ],
    "Masstige": [
        "CeraVe", "ラ ロッシュ ポゼ", "スキンシューティカルズ", "ヴィシー", "キュレル"
    ],
    "Mass": [
        "ロレアル パリ", "メイベリン", "NYX", "ガルニエ",
        "ダヴ", "シンプル", "ビオレ", "カネボウ"
    ],
}

TIER_ORDER = ["Luxury", "Prestige", "Masstige", "Mass"]

In [ ]:
# ── Cell 4: YouTube取得関数 ────────────────────────────────
from datetime import datetime, timedelta

def get_video_ids_jp(keyword, max_videos=10, years_back=5):
    """日本語キーワードでYouTube動画を検索し、動画IDを返す"""
    published_after = (
        datetime.now() - timedelta(days=365 * years_back)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    request = youtube.search().list(
        q=keyword + " レビュー コスメ",   # "review cosmetics" in Japanese
        part="id",
        type="video",
        maxResults=max_videos,
        relevanceLanguage="ja",           # Japanese results only
        regionCode="JP",                  # Japan region
        publishedAfter=published_after
    )
    response = request.execute()
    return [item["id"]["videoId"] for item in response.get("items", [])]


def get_comments_jp(video_id, max_comments=100):
    """動画のトップコメントを取得する"""
    comments = []
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=max_comments,
            order="relevance"
        )
        response = request.execute()
        for item in response.get("items", []):
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "text":     snippet["textDisplay"],
                "likes":    snippet["likeCount"],
                "date":     snippet["publishedAt"],
                "video_id": video_id,
            })
    except Exception as e:
        print(f"  コメント取得失敗 {video_id}: {e}")
    return comments

In [ ]:
# ── Cell 5: 日本語センチメント関数 ────────────────────────────

import spacy
nlp = spacy.load("ja_ginza")

# ── 感情極性辞書（絵文字＋形容詞パターン）─────────────────────
POSITIVE_EMOJI = set("😍💕✨🥰💖💗💓💞😊🙌👍💯🌟⭐🎉🫶❤️🩷🩵💜💙💚💛🧡❤")
NEGATIVE_EMOJI = set("😢😭😤😠👎💔😞😔🤢😩😫😰😨")

POSITIVE_PATTERNS = {"好き", "良い", "いい", "最高", "おすすめ", "気持ちいい",
                     "綺麗", "きれい", "可愛い", "かわいい", "嬉しい", "うれしい",
                     "素晴らしい", "すばらしい", "潤う", "うるおう", "効果",
                     "リピ", "リピート", "感動", "満足", "優しい", "やさしい"}

NEGATIVE_PATTERNS = {"悪い", "最悪", "ひどい", "がっかり", "失望", "合わない",
                     "かゆい", "荒れる", "べたべた", "重い", "臭い", "くさい",
                     "高い", "無駄", "効果なし", "損", "後悔"}


def score_japanese(text):
    """
    GiNZA形態素解析 + 絵文字パターンで感情スコアを算出
    戻り値: -1.0（最もネガティブ）〜 +1.0（最もポジティブ）
    """
    if not text or len(text.strip()) == 0:
        return 0.0

    pos_count = 0
    neg_count = 0

    # ── 絵文字スコアリング ─────────────────────────────────
    for char in text:
        if char in POSITIVE_EMOJI:
            pos_count += 1
        elif char in NEGATIVE_EMOJI:
            neg_count += 1

    # ── GiNZA形態素解析 ────────────────────────────────────
    doc = nlp(text[:500])   # 長文は500文字に制限（速度対策）
    for token in doc:
        # 形容詞・感動詞・名詞のポジティブ/ネガティブパターン照合
        if token.lemma_ in POSITIVE_PATTERNS:
            pos_count += 2   # 語彙マッチは絵文字より重みを高く
        elif token.lemma_ in NEGATIVE_PATTERNS:
            neg_count += 2

    # ── 正規化スコア算出 ───────────────────────────────────
    total = pos_count + neg_count
    if total == 0:
        return 0.0   # シグナルなし → ニュートラル

    return (pos_count - neg_count) / total

In [ ]:
# ── Cell 6: メインデータ取得 ───────────────────────────────
import os, pickle

CACHE_FILE = "../data/processed/youtube_comments_jp_raw.pkl"
USE_CACHE = True  # ← Falseにすると新規取得（APIクォータ消費）

if USE_CACHE and os.path.exists(CACHE_FILE):
    print("キャッシュから読み込み中...")
    with open(CACHE_FILE, "rb") as f:
        df_comments_jp = pickle.load(f)
    print(f"読み込み完了: {len(df_comments_jp)}件")

else:
    records = []

    for tier, keywords in TIER_KEYWORDS_JP.items():
        for keyword in keywords:
            print(f"取得中: {keyword}...")
            video_ids = get_video_ids_jp(keyword, max_videos=10, years_back=5)
            time.sleep(0.5)

            for vid_id in video_ids:
                comments = get_comments_jp(vid_id, max_comments=100)
                time.sleep(0.5)

                for c in comments:
                    score = score_japanese(c["text"])
                    records.append({
                        "tier":     tier,
                        "keyword":  keyword,
                        "text":     c["text"],
                        "likes":    c["likes"],
                        "date":     c["date"],
                        "compound": score,
                    })

    df_comments_jp = pd.DataFrame(records)
    print(f"\n取得完了: {len(df_comments_jp)}件のコメント")
    print(df_comments_jp["tier"].value_counts())

    with open(CACHE_FILE, "wb") as f:
        pickle.dump(df_comments_jp, f)
    print("キャッシュ保存完了 ✓")

In [ ]:
# ── Cell 7: キャッシュ保存 ─────────────────────────────────
import pickle
import json
from datetime import datetime

# ── 生データ保存 ───────────────────────────────────────────
with open("../data/processed/youtube_comments_jp_raw.pkl", "wb") as f:
    pickle.dump(df_comments_jp, f)
print("保存完了: youtube_comments_jp_raw.pkl")

# ── メタデータ保存 ─────────────────────────────────────────
pull_metadata_jp = {
    "pulled_at":       datetime.now().isoformat(),
    "total_comments":  len(df_comments_jp),
    "date_range_min":  pd.to_datetime(df_comments_jp["date"]).min().isoformat(),
    "date_range_max":  pd.to_datetime(df_comments_jp["date"]).max().isoformat(),
    "tier_counts":     df_comments_jp["tier"].value_counts().to_dict(),
}

with open("../data/processed/youtube_pull_metadata_jp.json", "w", encoding="utf-8") as f:
    json.dump(pull_metadata_jp, f, indent=2, ensure_ascii=False)

print(f"取得日時:     {pull_metadata_jp['pulled_at']}")
print(f"コメント期間: {pull_metadata_jp['date_range_min']} → {pull_metadata_jp['date_range_max']}")
print(f"合計:         {pull_metadata_jp['total_comments']}件")

In [ ]:
# ── Cell 8: 集計 ───────────────────────────────────────────
TIER_ORDER = ["Luxury", "Prestige", "Masstige", "Mass"]

def weighted_mean(group):
    weights = group["likes"] + 1
    return (group["compound"] * weights).sum() / weights.sum()

tier_sentiment_jp = (
    df_comments_jp
    .groupby("tier")
    .apply(weighted_mean)
    .reindex(TIER_ORDER)
    .reset_index()
)
tier_sentiment_jp.columns = ["Tier", "Weighted_Sentiment"]

tier_volume_jp = (
    df_comments_jp
    .groupby("tier")
    .size()
    .reindex(TIER_ORDER)
    .reset_index()
)
tier_volume_jp.columns = ["Tier", "Comment_Count"]

tier_sentiment_jp.to_csv("../data/processed/youtube_sentiment_tier_jp.csv", index=False)

print(tier_sentiment_jp.to_string(index=False))
print()
print(tier_volume_jp.to_string(index=False))

In [ ]:
# ── Cell 9: チャート1 センチメントスコア ──────────────────
import matplotlib
import matplotlib.font_manager as fm

# ── 日本語フォント設定 ────────────────────────────────────────
jp_fonts = ["MS Gothic", "Yu Gothic", "Meiryo", "IPAGothic", "Noto Sans CJK JP"]
available = [f.name for f in fm.fontManager.ttflist]

for font in jp_fonts:
    if font in available:
        plt.rcParams["font.family"] = font
        print(f"日本語フォント設定: {font}")
        break
else:
    print("警告: 日本語フォントが見つかりません — 文字化けの可能性あり")
    print("利用可能なフォント候補:")
    print([f for f in available if any(x in f for x in ["Gothic", "Mincho", "CJK", "Noto"])][:10])

# ── チャート1: 発散型ドットプロット — YouTube JP ────────────
fig, ax = plt.subplots(figsize=(8, 5))

scores_jp = tier_sentiment_jp.set_index("Tier")["Weighted_Sentiment"].reindex(TIER_ORDER)

for i, (tier, val) in enumerate(scores_jp.items()):
    ax.plot([0, val], [i, i], color=TIER_COLOURS[tier], linewidth=2.5, alpha=0.7)
    ax.scatter(val, i, color=TIER_COLOURS[tier], s=180, zorder=5)
    ax.text(val + 0.012, i, f"{val:.3f}", va="center", fontsize=10, fontweight="bold")

ax.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.6)
ax.set_yticks(range(len(TIER_ORDER)))
ax.set_yticklabels(TIER_ORDER, fontsize=11)
ax.set_xlabel("加重スコア (-1 〜 +1)", fontsize=10)
ax.set_title("加重平均センチメントスコア by ビューティティア\n(YouTube日本語コメント, GiNZA/Sudachi)",
             fontsize=11, fontweight="bold")
ax.set_xlim(-0.5, 0.5)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
save_chart(fig, "04b_sentiment_by_tier_jp.png")
plt.show()

### cosmeデータ分析

In [ ]:
# Cell 1 ── Library imports ───────────────────────────────

import re
import time
import unicodedata
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from difflib import SequenceMatcher

# User agent (important for scraping stability)
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
}

# Regex to extract /brands/<id>
BRAND_PATTERN = re.compile(r"/brands/(\d+)(?:/|$)")

# Resolver tuning
TOP_K = 12
MIN_SCORE = 0.45

In [ ]:
# ── Cell 2: ブランド名、類似名定義 ───────────────────────────────────

BRANDS = [
    "エスティローダー", "クリニーク", "ラ・メール", "トム フォード ビューティ",
    "ジョー マローン", "ボビイ ブラウン", "ディオール ビューティ", "ゲラン",
    "ジバンシィ ビューティ", "ベネフィット", "フレッシュ",
    "ランコム", "イヴ・サンローラン ビューティ", "ジョルジオ アルマーニ ビューティ",
    "資生堂", "NARS", "ドランク エレファント", "クレ・ド・ポー ボーテ",
    "コスメデコルテ", "雪肌精", "ダーマロジカ",
    "CeraVe", "ラ ロッシュ ポゼ", "スキンシューティカルズ", "ヴィシー", "キュレル",
    "ロレアル パリ", "メイベリン", "NYX", "ガルニエ",
    "ダヴ", "シンプル", "ビオレ", "カネボウ"
]

ALIASES = {
    "ジョー マローン": ["ジョー マローン ロンドン", "Jo Malone London", "JoMaloneLondon"],
    "ジバンシィ ビューティ": ["ジバンシイ", "Givenchy"],
    "ジョルジオ アルマーニ ビューティ": ["アルマーニ ビューティ", "ジョルジオ アルマーニ", "GIORGIO ARMANI"],
    "ヴィシー": ["VICHY"],
    "NYX": ["NYX Professional Makeup", "NYXProfessionalMakeup"],
    "ダヴ": ["Dove"],
    "ガルニエ": ["garnier", "GARNIER"],
    "ダーマロジカ": ["Dermalogica"],
}

BLACKLIST = {
    "フレッシュ": ["プロフレッシュ"],
    "シンプル": ["シンプルアップ"],
}

In [ ]:
# ── Cell 3: ブランドID検索：Anchor-Based Resolver (Improved Ranking) ─────────

def _norm(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", "", s)
    return s.strip()

def _sim(a: str, b: str) -> float:
    return SequenceMatcher(None, _norm(a), _norm(b)).ratio()

def find_brand_page_anchor(original_keyword: str, query: str):
    search_url = f"https://www.cosme.net/search/?fw={quote_plus(query)}"
    r = requests.get(search_url, headers=HEADERS, timeout=20)

    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")
    candidates = []

    for a in soup.select("a[href]"):
        href = a["href"].strip()
        m = BRAND_PATTERN.search(href)
        if not m:
            continue

        brand_id = m.group(1)
        anchor_text = a.get_text(strip=True)
        if not anchor_text:
            continue

        candidates.append((brand_id, anchor_text))

    if not candidates:
        return None

    # Remove duplicate brand IDs
    seen = {}
    for bid, text in candidates:
        if bid not in seen:
            seen[bid] = text

    scored = []
    bad_terms = [_norm(x) for x in BLACKLIST.get(original_keyword, [])]

    for bid, text in list(seen.items())[:TOP_K]:
        nt = _norm(text)
        nq = _norm(query)

        # Blacklist filter
        if any(bt in nt for bt in bad_terms):
            continue

        # Base similarity
        score = _sim(query, text)

        # Boost if exact containment
        if nq in nt:
            score += 0.3

        scored.append((score, bid, text))

    if not scored:
        return None

    scored.sort(reverse=True)
    best_score, best_id, best_text = scored[0]

    if best_score < MIN_SCORE:
        return None

    return {
        "brand_id": best_id,
        "matched_name": best_text,
        "url": f"https://www.cosme.net/brands/{best_id}/review/",
        "confidence": round(best_score, 3),
        "query_used": query,
    }

def find_brand_with_aliases(keyword: str):
    queries = [keyword] + ALIASES.get(keyword, [])
    for q in queries:
        result = find_brand_page_anchor(keyword, q)
        time.sleep(0.6)
        if result:
            return result
    return None

In [ ]:
# ── Cell 4: ブランドページ確認（ガードキャッシュ付き） ─────────────

import os

CACHE_FILE = "../data/processed/brand_mapping.pkl"
RUN_BRAND_RESOLUTION = False   # ← switch: set True only when you want to scrape

if os.path.exists(CACHE_FILE) and not RUN_BRAND_RESOLUTION:
    print("Loading cached brand mapping...")
    
    brand_df = pd.read_pickle(CACHE_FILE)
    
    brand_mapping = {
        row["brand"]: {
            "brand_id": row["brand_id"],
            "matched_name": row["matched_name"],
            "url": row["url"],
            "confidence": row["confidence"],
            "query_used": row["query_used"],
        }
        for _, row in brand_df.iterrows()
        if row["status"] == "resolved"
    }

    not_found = brand_df.loc[brand_df["status"] == "not_found", "brand"].tolist()

else:
    print("Running brand resolution scraper...\n")

    brand_mapping = {}
    not_found = []
    results = []

    for i, brand in enumerate(BRANDS, 1):
        print(f"[{i}/{len(BRANDS)}] Resolving: {brand}")

        result = find_brand_with_aliases(brand)

        if result:
            brand_mapping[brand] = result
            results.append({
                "brand": brand,
                "brand_id": result["brand_id"],
                "matched_name": result["matched_name"],
                "url": result["url"],
                "confidence": result["confidence"],
                "query_used": result["query_used"],
                "status": "resolved"
            })
            print(f"  ✓ id={result['brand_id']} | conf={result['confidence']}")
        else:
            not_found.append(brand)
            results.append({
                "brand": brand,
                "brand_id": None,
                "matched_name": None,
                "url": None,
                "confidence": None,
                "query_used": None,
                "status": "not_found"
            })
            print("  ✗ Not found")

        time.sleep(0.6)

    brand_df = pd.DataFrame(results)

    print("\n── Summary ──")
    print("Resolved:", len(brand_mapping))
    print("Not Found:", len(not_found))

    brand_df.to_pickle(CACHE_FILE)
    print("\nCache saved:", CACHE_FILE)

brand_df.head()

In [ ]:
# ── Cell 5: ブランド×ティアマッピング ────────────────────────

BRAND_TIER = {
    "エスティローダー":           "Luxury",
    "クリニーク":                 "Luxury",
    "ラ・メール":                 "Luxury",
    "トム フォード ビューティ":    "Luxury",
    "ジョー マローン":            "Luxury",
    "ボビイ ブラウン":            "Luxury",
    "ディオール ビューティ":       "Luxury",
    "ゲラン":                    "Luxury",
    "ジバンシィ ビューティ":       "Luxury",
    "ベネフィット":               "Luxury",
    "フレッシュ":                 "Luxury",
    "ランコム":                  "Prestige",
    "イヴ・サンローラン ビューティ": "Prestige",
    "ジョルジオ アルマーニ ビューティ": "Prestige",
    "資生堂":                    "Prestige",
    "NARS":                     "Prestige",
    "ドランク エレファント":        "Prestige",
    "クレ・ド・ポー ボーテ":       "Prestige",
    "コスメデコルテ":             "Prestige",
    "雪肌精":                   "Prestige",
    "ダーマロジカ":               "Prestige",
    "CeraVe":                   "Masstige",
    "ラ ロッシュ ポゼ":           "Masstige",
    "スキンシューティカルズ":      "Masstige",
    "ヴィシー":                  "Masstige",
    "キュレル":                  "Masstige",
    "ロレアル パリ":              "Mass",
    "メイベリン":                "Mass",
    "NYX":                      "Mass",
    "ガルニエ":                  "Mass",
    "ダヴ":                     "Mass",
    "シンプル":                  "Mass",
    "ビオレ":                   "Mass",
    "カネボウ":                  "Mass",
}

In [ ]:
# ── Cell 6  @cosmeレビュースクレイパー─────────────────────────────────────────────

def scrape_brand_reviews(brand_name, brand_id, max_pages=5):
    records = []
    base = f"https://www.cosme.net/brands/{brand_id}/review/"

    for page in range(1, max_pages + 1):
        url = base if page == 1 else f"{base}?page={page}"
        response = requests.get(url, headers=HEADERS, timeout=20)
        time.sleep(1.5)

        if response.status_code != 200:
            print(f"  {brand_name} p{page}: {response.status_code} — 終了")
            break

        soup = BeautifulSoup(response.content, "html.parser")
        review_divs = soup.find_all("div", class_="reviewInformation")

        if not review_divs:
            print(f"  {brand_name} p{page}: レビューなし — 終了")
            break

        for div in review_divs:
            rating_tag = div.find(
                "p", class_=lambda x: x and "reviewer-rating" in str(x)
            )
            rating = None
            if rating_tag:
                raw = rating_tag.text.strip()
                if raw.isdigit():
                    rating = int(raw)

            text = next(
                (p.text.strip() for p in div.find_all("p")
                 if len(p.text.strip()) > 30),
                None
            )

            date_tag = div.find(
                "p", class_=lambda x: x and "date" in str(x).lower()
            )
            date = date_tag.text.strip() if date_tag else None

            if rating and text:
                records.append({
                    "brand":    brand_name,
                    "tier":     BRAND_TIER[brand_name],
                    "rating":   rating,
                    "compound": round((rating - 4) / 3, 4),
                    "text":     text,
                    "date":     date,
                })

        print(f"  {brand_name} p{page}: {len(review_divs)}件取得")

    return records

In [ ]:
# ── Cell 7: 全ブランド実行（キャッシュ対応）──────────────────
import os, pickle

COSME_CACHE = "../data/processed/cosme_reviews_raw.pkl"
USE_CACHE = True  # ← Falseにすると再スクレイピング（サーバー負荷注意）

if USE_CACHE and os.path.exists(COSME_CACHE):
    print("キャッシュから読み込み中...")
    with open(COSME_CACHE, "rb") as f:
        df_cosme = pickle.load(f)
    print(f"読み込み完了: {len(df_cosme)}件")

else:
    all_records = []

    for brand_name, info in brand_mapping.items():
        print(f"\n取得中: {brand_name} (id={info['brand_id']})")
        records = scrape_brand_reviews(
            brand_name, info["brand_id"], max_pages=5
        )
        all_records.extend(records)
        time.sleep(2)

    df_cosme = pd.DataFrame(all_records)
    print(f"\n取得完了: {len(df_cosme)}件のレビュー")
    print(df_cosme["tier"].value_counts())

    with open(COSME_CACHE, "wb") as f:
        pickle.dump(df_cosme, f)
    print("キャッシュ保存完了 ✓")

In [ ]:
# ── Cell 8: ティア別集計 ─────────────────────────────────────

cosme_sentiment = (
    df_cosme.groupby("tier")
    .agg(
        Mean_Rating=("rating", "mean"),
        Mean_Compound=("compound", "mean"),
        Review_Count=("rating", "count"),
    )
    .round(3)
    .reset_index()
)

print(cosme_sentiment)

# CSV保存
df_cosme.to_csv("../data/processed/cosme_reviews_raw.csv", index=False, encoding="utf-8-sig")
cosme_sentiment.to_csv("../data/processed/cosme_sentiment_tier.csv", index=False, encoding="utf-8-sig")
print("\n保存完了: cosme_reviews_raw.csv, cosme_sentiment_tier.csv")

In [ ]:
# ── @cosme センチメント可視化 ─────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams["font.family"] = "MS Gothic"

TIER_ORDER  = ["Luxury", "Prestige", "Masstige", "Mass"]
TIER_COLORS = {
    "Luxury":   "#C9A84C",
    "Prestige": "#7B9EA6",
    "Masstige": "#A8C5A0",
    "Mass":     "#C4A882",
}

cosme_ordered = cosme_sentiment.set_index("tier").reindex(TIER_ORDER).reset_index()

# ── @cosme センチメント（発散型ドットプロット）───────────────
fig, ax = plt.subplots(figsize=(8, 5))

cosme_ordered = cosme_sentiment.set_index("tier").reindex(TIER_ORDER).reset_index()
cosme_scores  = cosme_ordered.set_index("tier")["Mean_Compound"].to_dict()

for i, tier in enumerate(TIER_ORDER):
    val = cosme_scores[tier]
    ax.plot([0, val], [i, i], color=TIER_COLORS[tier], linewidth=2.5, alpha=0.7)
    ax.scatter(val, i, color=TIER_COLORS[tier], s=180, zorder=5)
    ax.text(val + 0.035, i, f"{val:.3f}", va="center", fontsize=10, fontweight="bold")

ax.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.6)
ax.set_yticks(range(len(TIER_ORDER)))
ax.set_yticklabels(TIER_ORDER, fontsize=11)
ax.set_xlabel("センチメントスコア（−1〜＋1）", fontsize=10)
ax.set_title("@cosme ティア別センチメントスコア\n（星評価ベース, -1〜+1変換）",
             fontsize=11, fontweight="bold")
ax.set_xlim(-0.6, 0.6)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../data/processed/04b_cosme_sentiment.png", dpi=150, bbox_inches="tight")
print("保存完了: 04b_cosme_sentiment.png")
plt.show()

In [ ]:
# ── ダンベルプロット：YouTube JP vs @cosme ────────────────
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(9, 5))

yt_jp_scores  = tier_sentiment_jp.set_index("Tier")["Weighted_Sentiment"].to_dict()
cosme_scores  = cosme_ordered.set_index("tier")["Mean_Compound"].to_dict()

YT_COLOR    = "#6A8EAE"
COSME_COLOR = "#C4956A"

for i, tier in enumerate(TIER_ORDER):
    yt_val    = yt_jp_scores.get(tier, 0)
    cosme_val = cosme_scores[tier]

    # Connecting line
    ax.plot([yt_val, cosme_val], [i, i],
            color="lightgrey", linewidth=2.5, zorder=1)

    # YouTube JP dot
    ax.scatter(yt_val, i, color=YT_COLOR, s=200, zorder=5, label="YouTube JP" if i == 0 else "")
    ax.text(yt_val - 0.035, i, f"{yt_val:.3f}",
            va="center", ha="right", fontsize=9, color=YT_COLOR, fontweight="bold")

    # @cosme dot
    ax.scatter(cosme_val, i, color=COSME_COLOR, s=200, zorder=5, label="@cosme" if i == 0 else "")
    ax.text(cosme_val + 0.035, i, f"{cosme_val:.3f}",
            va="center", ha="left", fontsize=9, color=COSME_COLOR, fontweight="bold")

    # Delta annotation
    delta = cosme_val - yt_val
    mid   = (yt_val + cosme_val) / 2
    ax.text(mid, i + 0.22, f"Δ{delta:+.3f}",
            va="bottom", ha="center", fontsize=8, color="grey")

ax.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.5)
ax.set_yticks(range(len(TIER_ORDER)))
ax.set_yticklabels(TIER_ORDER, fontsize=11)
ax.set_xlabel("センチメントスコア（−1〜＋1）", fontsize=10)
ax.set_title("YouTube JP vs @cosme：ソース間比較\n（ダンベルプロット）",
             fontsize=11, fontweight="bold")
ax.set_xlim(-0.1, 0.8)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=YT_COLOR,
           markersize=10, label="YouTube JP"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COSME_COLOR,
           markersize=10, label="@cosme"),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig("../data/processed/04b_cosme_yt_dumbbell.png", dpi=150, bbox_inches="tight")
print("保存完了: 04b_cosme_yt_dumbbell.png")
plt.show()

In [ ]:
# ── 動的主要観察結果 ──────────────────────────────────────
s_jp    = tier_sentiment_jp.set_index("Tier")["Weighted_Sentiment"]
s_cosme = cosme_sentiment.set_index("tier")["Mean_Compound"]

top_cosme    = s_cosme.idxmax()
bottom_cosme = s_cosme.idxmin()
convergence  = abs(s_jp["Masstige"] - s_cosme["Masstige"])

comparison_rows = "\n".join([
    f"| {t} | {s_jp[t]:.3f} | {s_cosme[t]:.3f} | {s_cosme[t] - s_jp[t]:+.3f} |"
    for t in TIER_ORDER
])

obs = f"""
## 主要観察結果

**センチメントスコア（@cosme星評価ベース, compound -1〜+1）:**
{chr(10).join([f"- {t}: {s_cosme[t]:.3f}" for t in TIER_ORDER])}
- 最高スコア: {top_cosme} ({s_cosme[top_cosme]:.3f})
- 最低スコア: {bottom_cosme} ({s_cosme[bottom_cosme]:.3f})

**ソース間比較（YouTube JP vs @cosme）:**

| ティア | YouTube JP | @cosme | 差異 |
|--------|-----------|--------|------|
{comparison_rows}

**マスティージの収束:**
YouTubeとatcosmeの差異: {convergence:.3f} — 本プロジェクト最強のクロスソース検証結果

**ランク順位の一貫性:**
両ソースでランク順位が完全一致。プラットフォーム特性の差異があっても
ティア間の相対的序列は堅牢であることが確認された。

**仮説への示唆**
- 全体的に、YouTubeのセンチメントスコアは@cosmeより著しく低く推移している。
（cosmeは定量化されている１～７★評価に対し、YouTubeはテキストレビューに基づく評価したから？）
  但し、Tier間の相対的な順位は両ソースで完全に一致しているため、両データソースが同様の消費者認識を反映している
  可能性が高い。
- Masstigeティアのスコアが両ソースで非常に近く（YouTube: {s_jp['Masstige']:.3f} vs @cosme: {s_cosme['Masstige']:.3f}）、
　また唯一、@cosmeよりもYouTubeのスコアが高い点も特に注目される。
- 日本の消費者はMasstigeブランドに対して特に好意的な認識を持っている傾向が明確に見られる。
"""
print(obs)